# 10 · Steady heat in the cup ☕

| 💻 Local | 🌐 Static | ▶ JupyterLite | ☁️ Colab |
|:--:|:--:|:--:|:--:|
| ✅ | [✅](/) | [✅](/lite/notebooks/index.html?path=10-steady-heat.ipynb) | [✅](https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/10-steady-heat.ipynb) |

<sub>✅ runs here · ⌛ runs but slowly (rough time)</sub>


Hot coffee warms the cup; the cup cools to the surrounding air. We solve the
**stationary heat equation** and answer a burning question: *where is it safe
to hold the cup?*

$$
-\nabla\!\cdot(\kappa\,\nabla T) = f \;\text{ in }\Omega,\qquad
\underbrace{-\kappa\,\partial_n T = \alpha\,(T-T_\infty)}_{\text{Newton cooling (Robin)}}
\text{ on }\Gamma_{\text{outer}},\qquad
\underbrace{\partial_n T = 0}_{\text{insulated (Neumann)}}\text{ on }\Gamma_{\text{bottom}} .
$$

Its weak form needs **no** Dirichlet data — the cooling term makes it
well-posed: find $T\in H^1$ with
$$
\int_\Omega \kappa\,\nabla T\!\cdot\!\nabla w
+ \int_{\Gamma_{\text{outer}}}\!\!\alpha\,T\,w
= \int_\Omega f\,w + \int_{\Gamma_{\text{outer}}}\!\!\alpha\,T_\infty\,w
\qquad\forall\,w .
$$

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from netgen.occ import WorkPlane, Axes, Z, Y, OCCGeometry
from ngsolve import *
from ngsolve.webgui import Draw

## 1. Geometry and boundary names

We reuse our coffee cup and label the boundary: the bottom sits on an
insulating coaster (`"bottom"`), the rest cools to the air (`"outer"`).

In [ ]:
def coffee_cup():
    Wb, Wt, H = 4.0, 5.0, 6.0
    body = (WorkPlane().MoveTo(-Wb/2, 0).LineTo(Wb/2, 0)
            .LineTo(Wt/2, H).LineTo(-Wt/2, H).Close().Face())
    cx, cy = Wt/2 + 0.2, H/2
    outer = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, 1.7).Face()
    inner = WorkPlane(Axes((cx, cy, 0), n=Z)).Circle(0, 0, 1.0).Face()
    return body + (outer - inner)

cup = coffee_cup()
cup.edges.name = "outer"
cup.edges.Min(Y).name = "bottom"          # the bottom edge -> insulated
mesh = Mesh(OCCGeometry(cup, dim=2).GenerateMesh(maxh=0.4))
mesh.Curve(3)
print("boundaries:", set(mesh.GetBoundaries()))

## 2. Data

Conductivity $\kappa$, heat-transfer coefficient $\alpha$, air temperature
$T_\infty$, and the heat input $f$ from the hot coffee (a warm blob in the
lower body).

In [ ]:
kappa = 1.0
alpha = 3.0                                # cooling strength
T_air = 20.0
coffee = 45 * exp(-((x)**2 + (y - 2.2)**2) / 2.5)     # the hot coffee
Draw(coffee, mesh)

## 3. Assemble and solve

The Robin term lives on the boundary: it appears in **both** the bilinear form
(`alpha*u*v*ds("outer")`) and the right-hand side (`alpha*T_air*v*ds("outer")`).

In [ ]:
fes = H1(mesh, order=3)
u, v = fes.TnT()
a = BilinearForm(kappa*grad(u)*grad(v)*dx + alpha*u*v*ds("outer")).Assemble()
f = LinearForm(coffee*v*dx + alpha*T_air*v*ds("outer")).Assemble()

gfu = GridFunction(fes)
gfu.vec.data = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky") * f.vec
Draw(gfu, mesh)

## 4. Where to hold it?

We read off the temperature in the body and on the handle.

In [ ]:
print(f"coffee (body)  ≈ {gfu(mesh(0, 2.2)):5.1f} °C")
print(f"handle         ≈ {gfu(mesh(4.0, 3.0)):5.1f} °C")
print(f"→ grab the handle!  (air is {T_air:.0f} °C)")

## 5. A first postprocessing: heat loss

The total heat leaving through the cooling surface is
$\int_{\Gamma_{\text{outer}}}\alpha\,(T-T_\infty)\,ds$. In steady state it must
balance the heat put in by the coffee, $\int_\Omega f\,dx$ — a nice sanity
check.

In [ ]:
heat_in = Integrate(coffee, mesh)
heat_out = Integrate(alpha*(gfu - T_air), mesh.Boundaries("outer"))
print(f"heat in  = {heat_in:6.2f}")
print(f"heat out = {heat_out:6.2f}   (should match)")

:::{dropdown} 🧠 Quiz — why didn't we need a Dirichlet condition?
With Newton cooling on (part of) the boundary, the bilinear form is already
coercive on all of $H^1$: the boundary term $\int_\Gamma \alpha\,u\,v\,ds$
controls the otherwise-undetermined constant. A pure-Neumann problem (insulated
everywhere) would be singular — the temperature would be defined only up to a
constant.
:::

Next: the **solver toolbox** — what `a.mat.Inverse(...)` really does, and faster
alternatives for big problems.